In [ ]:
# ==============================================================
# UNIVERSIDAD NACIONAL DE CHIMBORAZO
# Carrera: Ingeniería en Ciencia de Datos e Inteligencia Artificial
#
# PROYECTO DE INVESTIGACIÓN FORMATIVA
#
# TEMA:
# Desarrollo de un Sistema Predictivo para Evaluar la Probabilidad
# de Impago en Clientes Financieros mediante Redes Neuronales
#
# RESPONSABLE DEL MODELO:
# Dennis Becerra
#
# ALGORITMO:
# Red Neuronal Artificial (MLPClassifier)
#
# DATASET:
# Default of Credit Card Clients
# Fuente: UCI Machine Learning Repository
#
# Plataforma:
# Google Colab
# ==============================================================

# ==============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ==============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import warnings
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

warnings.filterwarnings("ignore")

# Configuración de gráficos
plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (10, 6)

# Mostrar todas las columnas
pd.set_option("display.max_columns", None)

print("==============================================")
print("LIBRERÍAS IMPORTADAS CORRECTAMENTE")
print("==============================================")

# ==============================================================
# 2. CARGAR DATASET
# ==============================================================
print("\nSeleccione el archivo:")
print("default of credit card clients.csv")

uploaded = files.upload()

if not uploaded:
    raise ValueError("No se subió ningún archivo. Por favor, ejecuta la celda nuevamente.")

# Obtener automáticamente el nombre del archivo cargado
nombre_archivo = list(uploaded.keys())[0]

# Leer el archivo omitiendo la primera fila conflictiva si viene con el formato clásico de UCI
try:
    df = pd.read_csv(nombre_archivo, skiprows=1)
    if 'ID' not in df.columns and 'LIMIT_BAL' not in df.columns:
        # Si al omitirla no cuadran las columnas, leer normal
        df = pd.read_csv(nombre_archivo)
except:
    df = pd.read_csv(nombre_archivo)

# Limpieza de espacios en los nombres de las columnas
df.columns = df.columns.str.strip()

print("\nDataset cargado correctamente.")
print("Archivo:", nombre_archivo)

# ==============================================================
# 3. INFORMACIÓN GENERAL
# ==============================================================
print("\n==============================================")
print("DIMENSIONES DEL DATASET")
print("==============================================")
print("Filas :", df.shape[0])
print("Columnas :", df.shape[1])

print("\n==============================================")
print("PRIMERAS FILAS")
print("==============================================")
display(df.head())

print("\n==============================================")
print("ÚLTIMAS FILAS")
print("==============================================")
display(df.tail())

# ==============================================================
# 4. INFORMACIÓN DEL DATASET
# ==============================================================
print("\n==============================================")
print("INFORMACIÓN GENERAL")
print("==============================================")
df.info()

# ==============================================================
# 5. ESTADÍSTICAS DESCRIPTIVAS
# ==============================================================
print("\n==============================================")
print("ESTADÍSTICAS DESCRIPTIVAS")
print("==============================================")
display(df.describe().T)

# ==============================================================
# 6. VERIFICAR VALORES NULOS
# ==============================================================
print("\n==============================================")
print("VALORES FALTANTES")
print("==============================================")
display(df.isnull().sum())

# ==============================================================
# 7. VERIFICAR DUPLICADOS
# ==============================================================
duplicados = df.duplicated().sum()

print("\n==============================================")
print("REGISTROS DUPLICADOS")
print("==============================================")
print("Duplicados encontrados:", duplicados)

if duplicados > 0:
    df = df.drop_duplicates()
    print("Duplicados eliminados correctamente.")
else:
    print("No existen registros duplicados.")

# ==============================================================
# 8. RENOMBRAR VARIABLES
# ==============================================================
# Identificar dinámicamente la columna objetivo por si varía el nombre
columna_target = None
for col in ['default payment next month', 'default_payment_next_month', 'Y', 'default']:
    if col in df.columns:
        columna_target = col
        break
if columna_target is None:
    columna_target = df.columns[-1] # Por defecto la última

dicc_renombrar = {
    'LIMIT_BAL': 'Limite_Credito',
    'SEX': 'Sexo',
    'EDUCATION': 'Educacion',
    'MARRIAGE': 'Estado_Civil',
    'AGE': 'Edad',
    'PAY_0': 'Pago_Mes_Actual', 'PAY_2': 'Pago_Mes_2', 'PAY_3': 'Pago_Mes_3',
    'PAY_4': 'Pago_Mes_4', 'PAY_5': 'Pago_Mes_5', 'PAY_6': 'Pago_Mes_6',
    'BILL_AMT1': 'Factura_Mes_1', 'BILL_AMT2': 'Factura_Mes_2', 'BILL_AMT3': 'Factura_Mes_3',
    'BILL_AMT4': 'Factura_Mes_4', 'BILL_AMT5': 'Factura_Mes_5', 'BILL_AMT6': 'Factura_Mes_6',
    'PAY_AMT1': 'Pago_Anterior_1', 'PAY_AMT2': 'Pago_Anterior_2', 'PAY_AMT3': 'Pago_Anterior_3',
    'PAY_AMT4': 'Pago_Anterior_4', 'PAY_AMT5': 'Pago_Anterior_5', 'PAY_AMT6': 'Pago_Anterior_6',
    columna_target: 'Incumplimiento'
}

df.rename(columns=dicc_renombrar, inplace=True)

print("\n==============================================")
print("VARIABLES RENOMBRADAS")
print("==============================================")
display(df.head())

# ==============================================================
# 9. TIPOS DE DATOS
# ==============================================================
print("\n==============================================")
print("TIPOS DE DATOS")
print("==============================================")
tipos = pd.DataFrame(df.dtypes, columns=["Tipo de dato"])
display(tipos)

# ==============================================================
# 10. VARIABLES NUMÉRICAS Y CATEGÓRICAS
# ==============================================================
variables_categoricas = ["Sexo", "Educacion", "Estado_Civil", "Incumplimiento"]
variables_numericas = []

for columna in df.columns:
    if columna not in variables_categoricas:
        variables_numericas.append(columna)

print("\n==============================================")
print("VARIABLES NUMÉRICAS")
print("==============================================")
print(variables_numericas)

print("\n==============================================")
print("VARIABLES CATEGÓRICAS")
print("==============================================")
print(variables_categoricas)

# ==============================================================
# 11. DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# ==============================================================
print("\n==============================================")
print("DISTRIBUCIÓN DEL INCUMPLIMIENTO")
print("==============================================")
display(df["Incumplimiento"].value_counts())

plt.figure(figsize=(7, 5))
sns.countplot(x="Incumplimiento", data=df, palette="Set2")
plt.title("Distribución de Clientes con y sin Incumplimiento")
plt.xlabel("Incumplimiento")
plt.ylabel("Cantidad de Clientes")
plt.show()

# ==============================================================
# 12. PORCENTAJE DE CADA CLASE
# ==============================================================
porcentaje = df["Incumplimiento"].value_counts(normalize=True) * 100
print("\n==============================================")
print("PORCENTAJE DE CADA CLASE")
print("==============================================")
display(round(porcentaje, 2))

# ==============================================================
# 13. RESUMEN DEL DATASET
# ==============================================================
print("\n==============================================")
print("RESUMEN")
print("==============================================")
print(f"""
Número de registros : {df.shape[0]}
Número de variables : {df.shape[1]}

Variable objetivo : Incumplimiento
0 = Cliente cumple sus pagos.
1 = Cliente incumplirá el pago del próximo mes.

El dataset se encuentra listo para iniciar el
Análisis Exploratorio de Datos (EDA).
""")

# ==============================================================
# PARTE 2A: ANÁLISIS EXPLORATORIO DE DATOS (EDA)
# ==============================================================
print("="*80)
print("ANÁLISIS EXPLORATORIO DE DATOS (EDA)")
print("="*80)

print("\nGenerando histogramas...")
vars_num_disp = df.select_dtypes(include=["int64", "float64"]).columns
df[vars_num_disp].hist(figsize=(22, 20), bins=25, edgecolor="black")
plt.suptitle("Distribución de las Variables Numéricas", fontsize=20, fontweight="bold")
plt.tight_layout()
plt.show()

# Distribución del Límite de Crédito
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="Limite_Credito", bins=30, kde=True, color="royalblue")
plt.title("Distribución del Límite de Crédito")
plt.xlabel("Límite de Crédito")
plt.ylabel("Frecuencia")
plt.show()

# Distribución de la Edad
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="Edad", bins=30, kde=True, color="forestgreen")
plt.title("Distribución de la Edad")
plt.xlabel("Edad")
plt.ylabel("Cantidad")
plt.show()

# Sexo
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Sexo", palette="Set2")
plt.title("Distribución por Sexo")
plt.xlabel("Sexo")
plt.ylabel("Número de Clientes")
plt.show()
print("\nFrecuencia por Sexo")
display(df["Sexo"].value_counts())

# Educación
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Educacion", palette="Pastel1")
plt.title("Distribución por Nivel Educativo")
plt.xlabel("Educación")
plt.ylabel("Número de Clientes")
plt.show()
print("\nFrecuencia por Nivel Educativo")
display(df["Educacion"].value_counts())

# Estado Civil
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Estado_Civil", palette="Set3")
plt.title("Distribución por Estado Civil")
plt.xlabel("Estado Civil")
plt.ylabel("Número de Clientes")
plt.show()
print("\nFrecuencia Estado Civil")
display(df["Estado_Civil"].value_counts())

# Relaciones vs Incumplimiento
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Sexo", hue="Incumplimiento", palette="Dark2")
plt.title("Incumplimiento según Sexo")
plt.xlabel("Sexo")
plt.ylabel("Cantidad")
plt.legend(title="Incumplimiento")
plt.show()

plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="Educacion", hue="Incumplimiento", palette="tab10")
plt.title("Incumplimiento según Nivel Educativo")
plt.xlabel("Educación")
plt.ylabel("Cantidad")
plt.legend(title="Incumplimiento")
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Estado_Civil", hue="Incumplimiento", palette="Accent")
plt.title("Incumplimiento según Estado Civil")
plt.xlabel("Estado Civil")
plt.ylabel("Cantidad")
plt.legend(title="Incumplimiento")
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="Incumplimiento", y="Edad", palette="Set2")
plt.title("Edad según Incumplimiento")
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="Incumplimiento", y="Limite_Credito", palette="Set3")
plt.title("Límite de Crédito según Incumplimiento")
plt.show()

# ==============================================================
# PARTE 2B: ANÁLISIS EXPLORATORIO AVANZADO
# ==============================================================
print("="*80)
print("PARTE 2B - ANÁLISIS EXPLORATORIO AVANZADO")
print("="*80)

variables_pago = ["Pago_Mes_Actual", "Pago_Mes_2", "Pago_Mes_3", "Pago_Mes_4", "Pago_Mes_5", "Pago_Mes_6"]
plt.figure(figsize=(15, 6))
sns.boxplot(data=df[variables_pago], palette="Set2")
plt.title("Variables de Historial de Pago")
plt.xticks(rotation=30)
plt.show()

variables_factura = ["Factura_Mes_1", "Factura_Mes_2", "Factura_Mes_3", "Factura_Mes_4", "Factura_Mes_5", "Factura_Mes_6"]
plt.figure(figsize=(15, 6))
sns.boxplot(data=df[variables_factura], palette="Pastel1")
plt.title("Montos Facturados")
plt.xticks(rotation=30)
plt.show()

variables_pago_real = ["Pago_Anterior_1", "Pago_Anterior_2", "Pago_Anterior_3", "Pago_Anterior_4", "Pago_Anterior_5", "Pago_Anterior_6"]
plt.figure(figsize=(15, 6))
sns.boxplot(data=df[variables_pago_real], palette="Set3")
plt.title("Pagos Realizados")
plt.xticks(rotation=30)
plt.show()

print("="*80)
print("MATRIZ DE CORRELACIÓN")
print("="*80)
correlacion = df.corr(numeric_only=True)

plt.figure(figsize=(20, 15))
sns.heatmap(correlacion, cmap="coolwarm", annot=False, linewidths=0.3)
plt.title("Mapa de Correlación")
plt.show()

corr_incumplimiento = correlacion["Incumplimiento"].sort_values(ascending=False)
print("="*80)
print("CORRELACIÓN CON LA VARIABLE OBJETIVO")
print("="*80)
display(corr_incumplimiento)

top10 = corr_incumplimiento.drop("Incumplimiento").abs().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 6))
top10.sort_values().plot.barh(color="steelblue")
plt.title("Top 10 Variables más Relacionadas con el Incumplimiento")
plt.xlabel("Correlación Absoluta")
plt.show()

variables_top = top10.index.tolist()
variables_top.append("Incumplimiento")
plt.figure(figsize=(12, 8))
sns.heatmap(df[variables_top].corr(), annot=True, cmap="RdBu_r", fmt=".2f")
plt.title("Correlación de Variables más Importantes")
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="Edad", y="Limite_Credito", hue="Incumplimiento", alpha=0.5)
plt.title("Edad vs Límite de Crédito")
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="Factura_Mes_1", y="Pago_Anterior_1", hue="Incumplimiento", alpha=0.5)
plt.title("Factura Mes 1 vs Pago Mes 1")
plt.show()

# ==============================================================
# PARTE 3A: PREPROCESAMIENTO DE LOS DATOS
# ==============================================================
print("="*80)
print("INICIANDO PREPROCESAMIENTO DEL DATASET")
print("="*80)

if "ID" in df.columns:
    df.drop("ID", axis=1, inplace=True)
    print("✔ Columna ID eliminada.")

print("\nDimensiones actuales del dataset:", df.shape)

print("\n" + "="*80)
print("REVISIÓN DE VARIABLES CATEGÓRICAS")
print("="*80)
for variable in ["Sexo", "Educacion", "Estado_Civil"]:
    print(f"\n{variable}")
    print(sorted(df[variable].unique()))

print("\n" + "="*80)
print("CORRIGIENDO VARIABLE EDUCACIÓN Y ESTADO CIVIL")
print("="*80)
df["Educacion"] = df["Educacion"].replace({0: 4, 5: 4, 6: 4})
df["Estado_Civil"] = df["Estado_Civil"].replace({0: 3})

print("Valores finales Educación:", sorted(df["Educacion"].unique()))
print("Valores finales Estado Civil:", sorted(df["Estado_Civil"].unique()))

print("\n" + "="*80)
print("SEPARANDO VARIABLES")
print("="*80)
X = df.drop("Incumplimiento", axis=1)
y = df["Incumplimiento"]

print("Variables predictoras:", X.shape)
print("Variable objetivo:", y.shape)

print("\n" + "="*80)
print("DIVISIÓN DEL DATASET")
print("="*80)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

print("\n" + "="*80)
print("ESCALANDO VARIABLES")
print("="*80)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

joblib.dump(scaler, "standard_scaler.pkl")
print("✔ StandardScaler guardado correctamente.")

# ==============================================================
# PARTE 3B: VALIDACIÓN Y PREPARACIÓN FINAL DEL DATASET
# ==============================================================
print("="*80)
print("VALIDACIÓN DEL PREPROCESAMIENTO")
print("="*80)

plt.figure(figsize=(18, 6))
plt.boxplot(X_train_scaled.iloc[:, 0:10])
plt.xticks(range(1, 11), X_train_scaled.columns[:10], rotation=45)
plt.title("Variables Escaladas")
plt.show()

# Guardar los conjuntos preprocesados físicamente
X_train_scaled.to_csv("X_train_scaled.csv", index=False)
X_test_scaled.to_csv("X_test_scaled.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)
print("\nArchivos preprocesados guardados correctamente.")

# ==============================================================
# PARTE 4A: CONSTRUCCIÓN Y ENTRENAMIENTO DE LA RED NEURONAL (MLP)
# ==============================================================
print("="*80)
print("ENTRENAMIENTO DE LA RED NEURONAL ARTIFICIAL")
print("="*80)

print("\nCreando arquitecturas...")
# Solución definitiva al error de truncamiento completando los hiperparámetros obligatorios
modelo_1 = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    learning_rate_init=0.001,
    alpha=0.0001,
    max_iter=50,
    random_state=42,
    verbose=False
)

print("\n--> Iniciando el entrenamiento del modelo sobre el conjunto escalado...")
start_time = time.time()
modelo_1.fit(X_train_scaled, y_train)
tiempo_ejecucion = time.time() - start_time

print(f"[ÉXITO] Red Neuronal entrenada en: {tiempo_ejecucion:.2f} segundos.")
joblib.dump(modelo_1, "mlp_credit_model.pkl")

# ==============================================================
# PARTE 4B: EVALUACIÓN, REPORTE DE MÉTRICAS Y GRÁFICOS FINALES
# ==============================================================
print("\n" + "="*80)
print("EVALUACIÓN DE RENDIMIENTO DEL SISTEMA PREDICTIVO")
print("="*80)

# Generación de inferencias y cálculo probabilístico
y_pred = modelo_1.predict(X_test_scaled)
y_probs = modelo_1.predict_proba(X_test_scaled)[:, 1]

print("\nTABLA 1: REPORTE DE EVALUACIÓN MULTICLASE")
print("-" * 60)
print(classification_report(y_test, y_pred, target_names=['Cumple (Clase 0)', 'Impago (Clase 1)']))

# Métricas Globales
cm = confusion_matrix(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

print("\nTABLA 2: RESUMEN DE INDICADORES EN PRUEBA")
print("-" * 60)
df_resumen = pd.DataFrame({
    'Indicador Clave': ['Exactitud Global (Accuracy)', 'Área Bajo la Curva (AUC ROC)', 'Muestras de Validación'],
    'Desempeño Obtenido': [f"{acc:.4f}", f"{roc_auc:.4f}", f"{X_test_scaled.shape[0]} registros"]
})
display(df_resumen)

# Despliegue de los Gráficos de Evaluación
print("\n" + "="*80)
print("DESPLEGANDO COMPONENTES VISUALES")
print("="*80)
plt.figure(figsize=(16, 5))

# Subplot 1: Curva de Pérdida (Loss Curve)
plt.subplot(1, 2, 1)
plt.plot(modelo_1.loss_curve_, color='crimson', linewidth=2.5, label='Pérdida (Cross-Entropy)')
plt.title('Función de Pérdida de la Red Neuronal por Época', fontsize=12, fontweight='bold')
plt.xlabel('Iteraciones (Épocas)')
plt.ylabel('Valor de Pérdida')
plt.grid(True, linestyle='--')
plt.legend()

# Subplot 2: Matriz de Confusión Estilizada
plt.subplot(1, 2, 2)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Cumple (0)', 'Impago (1)'],
            yticklabels=['Cumple (0)', 'Impago (1)'])
plt.title('Matriz de Confusión Financiera Final', fontsize=12, fontweight='bold')
plt.xlabel('Predicción del Clasificador MLP')
plt.ylabel('Historial Financiero Real (UCI)')

plt.tight_layout()
plt.show()

# Gráfico Independiente 3: Curva ROC
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (Área = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.title('Curva ROC del Sistema Predictivo Financiero', fontsize=12, fontweight='bold')
plt.xlabel('Tasa de Falsos Positivos (1 - Especificidad)')
plt.ylabel('Tasa de Verdaderos Positivos (Sensibilidad)')
plt.legend(loc="lower right")
plt.grid(True, linestyle=':')
plt.show()

print("\n==============================================================")
print("PROCESO FINALIZADO CON ÉXITO: TODO LISTO PARA LA SUSTENTACIÓN")
print("==============================================================")

LIBRERÍAS IMPORTADAS CORRECTAMENTE

Seleccione el archivo:
default of credit card clients.csv
